In [17]:
!pip install imbalanced-learn

In [18]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

In [3]:
df = pd.read_csv('../data/raw/Thyroid_Diff.csv')
df.shape

(383, 17)

In [4]:
df['Recurred'] = df['Recurred'].map({'No': 0, 'Yes': 1})
df['Recurred'].value_counts()

Recurred
0    275
1    108
Name: count, dtype: int64

In [5]:
df['Gender'] = df['Gender'].map({'F': 0, 'M': 1})

for col in ['Smoking', 'Hx Smoking', 'Hx Radiothreapy']:
    df[col] = df[col].map({'No': 0, 'Yes': 1})

df['Focality'] = df['Focality'].map({'Uni-Focal': 0, 'Multi-Focal': 1})
df['M'] = df['M'].map({'M0': 0, 'M1': 1})

df[['Gender','Smoking','Hx Smoking','Hx Radiothreapy','Focality','M']].isnull().sum()

Gender             0
Smoking            0
Hx Smoking         0
Hx Radiothreapy    0
Focality           0
M                  0
dtype: int64

In [6]:
risk_order = [['Low', 'Intermediate', 'High']]
t_order = [['T1a', 'T1b', 'T2', 'T3a', 'T3b', 'T4a', 'T4b']]
n_order = [['N0', 'N1a', 'N1b']]
stage_order = [['I', 'II', 'III', 'IVA', 'IVB']]

df['Risk'] = OrdinalEncoder(categories=risk_order).fit_transform(df[['Risk']])
df['T'] = OrdinalEncoder(categories=t_order).fit_transform(df[['T']])
df['N'] = OrdinalEncoder(categories=n_order).fit_transform(df[['N']])
df['Stage'] = OrdinalEncoder(categories=stage_order).fit_transform(df[['Stage']])

df[['Risk', 'T', 'N', 'Stage']].head(10)

,Risk,T,N,Stage
0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0
6,0.0,0.0,0.0,0.0
7,0.0,0.0,0.0,0.0
8,0.0,0.0,0.0,0.0
9,0.0,0.0,0.0,0.0


In [7]:
df['Adenopathy'].value_counts()

Adenopathy
No           277
Right         48
Bilateral     32
Left          17
Extensive      7
Posterior      2
Name: count, dtype: int64

In [8]:
adenopathy_map = {
    'No': 'No',
    'Right': 'Unilateral',
    'Left': 'Unilateral',
    'Bilateral': 'Bilateral_Extensive',
    'Extensive': 'Bilateral_Extensive',
    'Posterior': 'Bilateral_Extensive'
}
df['Adenopathy'] = df['Adenopathy'].map(adenopathy_map)
df['Adenopathy'].value_counts()

Adenopathy
No                     277
Unilateral              65
Bilateral_Extensive     41
Name: count, dtype: int64

In [9]:
nominal_cols = ['Thyroid Function', 'Physical Examination', 'Adenopathy', 'Pathology', 'Response']
df = pd.get_dummies(df, columns=nominal_cols, drop_first=False)
df.shape

(383, 33)

In [10]:
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 383 entries, 0 to 382
Data columns (total 33 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   Age                                               383 non-null    int64  
 1   Gender                                            383 non-null    int64  
 2   Smoking                                           383 non-null    int64  
 3   Hx Smoking                                        383 non-null    int64  
 4   Hx Radiothreapy                                   383 non-null    int64  
 5   Focality                                          383 non-null    int64  
 6   Risk                                              383 non-null    float64
 7   T                                                 383 non-null    float64
 8   N                                                 383 non-null    float64
 9   M                    

In [11]:
df.isnull().sum().sum()   # phải ra 0

np.int64(0)

In [ ]:
df.to_csv('../data/processed/thyroid_processed.csv', index=False)

## Train test split ##

In [12]:
X = df.drop('Recurred', axis=1)
y = df['Recurred']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (306, 32)
Test shape: (77, 32)


In [13]:
print("Tỷ lệ trong train:")
print(y_train.value_counts(normalize=True))

print("\nTỷ lệ trong test:")
print(y_test.value_counts(normalize=True))

Tỷ lệ trong train:
Recurred
0    0.718954
1    0.281046
Name: proportion, dtype: float64

Tỷ lệ trong test:
Recurred
0    0.714286
1    0.285714
Name: proportion, dtype: float64


In [16]:
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

In [19]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Trước SMOTE:", y_train.value_counts().to_dict())
print("Sau SMOTE:", y_train_smote.value_counts().to_dict())

Trước SMOTE: {0: 220, 1: 86}
Sau SMOTE: {1: 220, 0: 220}


## Model dùng class_weight, train trên X_train/y_train ##

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

model_weighted = LogisticRegression(class_weight='balanced', random_state=42)
model_weighted.fit(X_train, y_train)

C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_weighted = LogisticRegression(class_weight='balanced', random_state=42)
model_weighted.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [24]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score
import matplotlib.pyplot as plt

## Train Logistic Regression ##

In [34]:
X_train_smote_lr, y_train_smote_lr = smote.fit_resample(X_train_scaled, y_train)

X_train_smote_rf, y_train_smote_rf = smote.fit_resample(X_train, y_train)

In [25]:
log_reg_weighted = LogisticRegression(class_weight='balanced', random_state=42)
log_reg_weighted.fit(X_train_scaled, y_train)

y_pred_lr_weighted = log_reg_weighted.predict(X_test_scaled)
print("=== Logistic Regression (class_weight) ===")
print(classification_report(y_test, y_pred_lr_weighted))

=== Logistic Regression (class_weight) ===
              precision    recall  f1-score   support

           0       0.96      1.00      0.98        55
           1       1.00      0.91      0.95        22

    accuracy                           0.97        77
   macro avg       0.98      0.95      0.97        77
weighted avg       0.97      0.97      0.97        77



In [29]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

log_reg_smote = LogisticRegression(random_state=42)
log_reg_smote.fit(X_train_smote, y_train_smote)

# Predict trên X_test_scaled — khớp đúng loại dữ liệu
y_pred_lr_smote = log_reg_smote.predict(X_test_scaled)
print("=== Logistic Regression (SMOTE) ===")
print(classification_report(y_test, y_pred_lr_smote))

=== Logistic Regression (SMOTE) ===
              precision    recall  f1-score   support

           0       0.96      1.00      0.98        55
           1       1.00      0.91      0.95        22

    accuracy                           0.97        77
   macro avg       0.98      0.95      0.97        77
weighted avg       0.97      0.97      0.97        77



In [30]:
from sklearn.ensemble import RandomForestClassifier

rf_weighted = RandomForestClassifier(class_weight='balanced', random_state=42)
rf_weighted.fit(X_train, y_train)

y_pred_rf_weighted = rf_weighted.predict(X_test)
print("=== Random Forest (class_weight) ===")
print(classification_report(y_test, y_pred_rf_weighted))

=== Random Forest (class_weight) ===
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        55
           1       1.00      0.86      0.93        22

    accuracy                           0.96        77
   macro avg       0.97      0.93      0.95        77
weighted avg       0.96      0.96      0.96        77



In [32]:
from imblearn.over_sampling import SMOTE

smote_rf = SMOTE(random_state=42)
X_train_smote_rf, y_train_smote_rf = smote_rf.fit_resample(X_train, y_train)

rf_smote = RandomForestClassifier(random_state=42)
rf_smote.fit(X_train_smote_rf, y_train_smote_rf)

y_pred_rf_smote = rf_smote.predict(X_test)
print("=== Random Forest (SMOTE) ===")
print(classification_report(y_test, y_pred_rf_smote))

=== Random Forest (SMOTE) ===
              precision    recall  f1-score   support

           0       0.96      1.00      0.98        55
           1       1.00      0.91      0.95        22

    accuracy                           0.97        77
   macro avg       0.98      0.95      0.97        77
weighted avg       0.97      0.97      0.97        77



In [35]:
# Tạo bộ X không có Response
response_cols = [c for c in X_train.columns if c.startswith('Response_')]
X_train_no_response = X_train.drop(columns=response_cols)
X_test_no_response = X_test.drop(columns=response_cols)

rf_no_response = RandomForestClassifier(class_weight='balanced', random_state=42)
rf_no_response.fit(X_train_no_response, y_train)

y_pred_no_response = rf_no_response.predict(X_test_no_response)
print("=== Random Forest (KHÔNG có Response) ===")
print(classification_report(y_test, y_pred_no_response))

=== Random Forest (KHÔNG có Response) ===
              precision    recall  f1-score   support

           0       0.95      0.96      0.95        55
           1       0.90      0.86      0.88        22

    accuracy                           0.94        77
   macro avg       0.93      0.91      0.92        77
weighted avg       0.93      0.94      0.93        77



In [36]:
import joblib

In [38]:
joblib.dump(rf_no_response, '../models/best_model.pkl')
print("Đã lưu model vào models/best_model.pkl")

Đã lưu model vào models/best_model.pkl


In [39]:
joblib.dump(X_train_no_response.columns.tolist(), '../models/feature_columns.pkl')
print("Đã lưu danh sách cột vào models/feature_columns.pkl")

Đã lưu danh sách cột vào models/feature_columns.pkl


In [40]:
encoding_info = {
    'risk_order': ['Low', 'Intermediate', 'High'],
    't_order': ['T1a', 'T1b', 'T2', 'T3a', 'T3b', 'T4a', 'T4b'],
    'n_order': ['N0', 'N1a', 'N1b'],
    'stage_order': ['I', 'II', 'III', 'IVA', 'IVB'],
    'adenopathy_map': {
        'No': 'No', 'Right': 'Unilateral', 'Left': 'Unilateral',
        'Bilateral': 'Bilateral_Extensive', 'Extensive': 'Bilateral_Extensive',
        'Posterior': 'Bilateral_Extensive'
    }
}
joblib.dump(encoding_info, '../models/encoding_info.pkl')

['../models/encoding_info.pkl']

In [41]:
loaded_model = joblib.load('../models/best_model.pkl')
loaded_columns = joblib.load('../models/feature_columns.pkl')

print("Số features model mong đợi:", len(loaded_columns))
print(loaded_columns)

# Thử predict lại để chắc chắn load đúng, kết quả phải giống y_pred_no_response
test_pred = loaded_model.predict(X_test_no_response)
print("Khớp với kết quả gốc:", (test_pred == y_pred_no_response).all())

Số features model mong đợi: 28
['Age', 'Gender', 'Smoking', 'Hx Smoking', 'Hx Radiothreapy', 'Focality', 'Risk', 'T', 'N', 'M', 'Stage', 'Thyroid Function_Clinical Hyperthyroidism', 'Thyroid Function_Clinical Hypothyroidism', 'Thyroid Function_Euthyroid', 'Thyroid Function_Subclinical Hyperthyroidism', 'Thyroid Function_Subclinical Hypothyroidism', 'Physical Examination_Diffuse goiter', 'Physical Examination_Multinodular goiter', 'Physical Examination_Normal', 'Physical Examination_Single nodular goiter-left', 'Physical Examination_Single nodular goiter-right', 'Adenopathy_Bilateral_Extensive', 'Adenopathy_No', 'Adenopathy_Unilateral', 'Pathology_Follicular', 'Pathology_Hurthel cell', 'Pathology_Micropapillary', 'Pathology_Papillary']
Khớp với kết quả gốc: True
